In [68]:
import os
import re
import numpy as np
import xarray as xr
import h5py
import pandas as pd

from scipy.spatial import cKDTree
from multiprocessing import Pool, cpu_count
from tqdm import tqdm


In [69]:
GPM_FOLDER = "/Users/utkarshpol/Desktop/project101/data/raw/gpm"
MERRA_FOLDER = "/Users/utkarshpol/Desktop/project101/data/raw/merra"
PROCESSED_FOLDER = "/Users/utkarshpol/Desktop/project101/data/processed"
CSV_FOLDER = "/Users/utkarshpol/Desktop/project101/data/parquet/csv"

In [70]:
def extract_date_from_gpm(filename):

    match = re.search(r"\.(\d{8})-S", filename)

    if match:
        return match.group(1)

    raise ValueError("Date not found")


In [71]:
def find_merra_file(date):

    for f in os.listdir(MERRA_FOLDER):

        if date in f and f.endswith(".nc4"):
            return os.path.join(MERRA_FOLDER,f)

    return None


In [72]:
def find_merra_file(date):

    for f in os.listdir(MERRA_FOLDER):

        if date in f and f.endswith(".nc4"):
            return os.path.join(MERRA_FOLDER,f)

    return None


In [73]:
TREE_CACHE = {}

In [74]:
def build_kdtree(merra_ds):

    lat = merra_ds.lat.values
    lon = merra_ds.lon.values

    lon2d , lat2d = np.meshgrid(lon,lat)

    grid_points = np.column_stack(

        (lat2d.ravel(),
         lon2d.ravel())

    )

    tree = cKDTree(grid_points)

    return tree , len(lon)


In [75]:
def build_scan_time(scan):

    year = scan["FS_ScanTime_Year"][:]
    month = scan["FS_ScanTime_Month"][:]
    day = scan["FS_ScanTime_DayOfMonth"][:]
    hour = scan["FS_ScanTime_Hour"][:]
    minute = scan["FS_ScanTime_Minute"][:]
    second = scan["FS_ScanTime_Second"][:]

    times = np.array([

        np.datetime64(

        f"{y:04d}-{m:02d}-{d:02d}"
        f"T{h:02d}:{mi:02d}:{s:02d}"

        )

        for y,m,d,h,mi,s in zip(
            year,month,day,hour,minute,second
        )

    ])

    return times


In [76]:
def process_gpm_file(gpm_path):

    filename = os.path.basename(gpm_path)

    print("Processing:",filename)

    date = extract_date_from_gpm(filename)

    merra_file = find_merra_file(date)

    if merra_file is None:

        print("No MERRA file:",date)
        return


    ###################################
    # LOAD MERRA
    ###################################

    merra = xr.open_dataset(
        merra_file,
        chunks={"time":1}
    )

    aod = merra["TOTEXTTAU"].values

    merra_time = merra.time.values


    ###################################
    # KD TREE CACHE
    ###################################

    global TREE_CACHE

    if "tree" not in TREE_CACHE:

        tree , lon_len = build_kdtree(merra)

        TREE_CACHE["tree"]=tree
        TREE_CACHE["lon_len"]=lon_len

    tree = TREE_CACHE["tree"]
    lon_len = TREE_CACHE["lon_len"]


    ###################################
    # LOAD DPR
    ###################################

    success = False


    with h5py.File(gpm_path,"r+") as f:

        fs = f["FS"]

        lat = fs["Latitude"][:]
        lon = fs["Longitude"][:]

        scan = fs["ScanTime"]

        scan_time = build_scan_time(scan)


        ##################################
        # TIME EXPAND
        ##################################

        cross = lat.shape[1]

        scan_time2d = np.repeat(
            scan_time[:,None],
            cross,
            axis=1
        )


        ##################################
        # FLATTEN
        ##################################

        lat_flat = lat.ravel()
        lon_flat = lon.ravel()

        scan_flat = scan_time2d.ravel()


        ##################################
        # SPATIAL MATCH
        ##################################

        points = np.column_stack(
            (lat_flat,lon_flat)
        )

        _,idx = tree.query(points)

        lat_index = idx // lon_len
        lon_index = idx % lon_len


        ##################################
        # TIME MATCH
        ##################################

        merra_time_int = (
        merra_time.astype("datetime64[m]")
        .astype(int)
        )

        scan_time_int = (
        scan_flat.astype("datetime64[m]")
        .astype(int)
        )

        diff = np.abs(

            scan_time_int[:,None]
            -
            merra_time_int

        )

        time_index = diff.argmin(axis=1)

        time_diff = diff[
            np.arange(len(time_index)),
            time_index
        ]

        valid_mask = time_diff <= 30


        ##################################
        # AOD EXTRACTION
        ##################################

        matched_aod = np.full(
            len(scan_flat),
            np.nan,
            dtype=np.float32
        )

        matched_aod[valid_mask] = aod[

            time_index[valid_mask],
            lat_index[valid_mask],
            lon_index[valid_mask]

        ]

        matched_aod = matched_aod.reshape(
            lat.shape
        )


        ##################################
        # SAVE DATASET  ✅ INSIDE WITH
        ##################################

        try:

            if "aod_matched" in fs:

                del fs["aod_matched"]

            fs.create_dataset(

                "aod_matched",

                data=matched_aod,

                compression="gzip"

            )

            success = True

            print("AOD written successfully")

        except Exception as e:

            print("AOD writing failed:", e)



    ##################################
    # CLOSE MERRA
    ##################################

    merra.close()


    ##################################
    # MOVE ONLY IF SUCCESS
    ##################################

    if success:

        import shutil

        new_path = os.path.join(

            PROCESSED_FOLDER,

            filename

        )

        shutil.move(

            gpm_path,
            new_path

        )

        print("Moved SUCCESSFULLY:",new_path)

    else:

        print("File NOT moved (AOD failed):",filename)

In [77]:
def export_csv(gpm_path):

    filename = os.path.basename(gpm_path)

    print("\nCSV:",filename)

    with h5py.File(gpm_path,"r") as f:

        fs = f["FS"]

        ################################
        # BASIC VARIABLES
        ################################

        lat = fs["Latitude"][:]

        lon = fs["Longitude"][:]

        R = fs["SLV"]["precipRateNearSurface"][:]

        Z = fs["SLV"]["zFactorFinalNearSurface"][:].astype(float)


        ################################
        # REMOVE DPR FILL VALUES
        ################################

        Z[ Z < -100 ] = np.nan


        ################################
        # LOAD DSD PROFILE
        ################################

        paramDSD_ds = fs["SLV"]["paramDSD"]

        paramDSD = paramDSD_ds[:]

        print("paramDSD shape:",paramDSD.shape)


        ################################
        # APPLY SCALE FACTOR
        ################################

        scale = paramDSD_ds.attrs.get(

            "scale_factor",

            1.0

        )

        paramDSD = paramDSD * scale


        ################################
        # NEAR SURFACE BIN
        ################################

        binBottom = fs["SLV"]["binEchoBottom"][:]


        ################################
        # CREATE OUTPUT ARRAYS
        ################################

        Dm = np.full(lat.shape,np.nan,dtype=float)

        DNBw = np.full(lat.shape,np.nan,dtype=float)


        ################################
        # EXTRACT NEAR SURFACE DSD
        ################################

        nscan , ncross = lat.shape

        for i in range(nscan):

            for j in range(ncross):

                b = binBottom[i,j]

                # VALID BIN CHECK

                if (b > 0) and (b < paramDSD.shape[2]):

                    DNBw[i,j] = paramDSD[i,j,b,0]

                    Dm[i,j] = paramDSD[i,j,b,1]


        ################################
        # LOAD AOD
        ################################

        aod = fs["aod_matched"][:]


        ################################
        # PRECIP FLAG
        ################################

        flag = fs["PRE"]["flagPrecip"][:]


        ################################
        # BUILD TIME ARRAY
        ################################

        scan = fs["ScanTime"]

        scan_time = build_scan_time(scan)

        cross = lat.shape[1]

        scan_time2d = np.repeat(

            scan_time[:,None],

            cross,

            axis=1

        )


        ################################
        # FILTER GOOD RAIN PIXELS
        ################################

        mask = (

            (flag == 1)

            &

            (~np.isnan(Z))

            &

            (R >= 0)

        )


        print("Rain Pixels:",np.sum(mask))


        ################################
        # BUILD DATAFRAME
        ################################

        df = pd.DataFrame({

            "Lat":lat[mask],

            "Lon":lon[mask],

            "R":R[mask],

            "Z":Z[mask],

            "Dm":Dm[mask],

            "DNBw":DNBw[mask],

            "time":

            scan_time2d[mask].astype(str),

            "AOD":

            aod[mask]

        })


        ################################
        # SAVE CSV
        ################################

        save_path = os.path.join(

            CSV_FOLDER,

            filename.replace(".HDF5",".csv")

        )

        df.to_csv(

            save_path,

            index=False

        )


        print("CSV rows:",df.shape)

        print("Saved:",save_path)

In [78]:
def main():

    gpm_files = [

        os.path.join(GPM_FOLDER,f)

        for f in os.listdir(GPM_FOLDER)

        if f.endswith(".HDF5")

    ]

    print("Total DPR files:",len(gpm_files))

    for gpm in tqdm(gpm_files):

        process_gpm_file(gpm)
    files = [

        os.path.join(PROCESSED_FOLDER,f)

        for f in os.listdir(PROCESSED_FOLDER)

        if f.endswith(".HDF5")

    ]

    print("Files:",len(files))

    for f in files:

        export_csv(f)

In [79]:
if __name__=="__main__":
    main()


Total DPR files: 1


  0%|          | 0/1 [00:00<?, ?it/s]

Processing: 2A.GPM.Ku.V9-20211125.20220606-S164656-E181924.046997.V07A.HDF5


KeyError: "Unable to synchronously open object (object 'FS_ScanTime_Year' doesn't exist)"

In [45]:
file = "/Users/utkarshpol/Desktop/project101/data/raw/gpm/2A.GPM.Ku.V9-20211125.20150101-S030204-E043435.004784.V07A.HDF5.nc4"


with h5py.File(file,"r") as f:


    print("Shape:", f.keys())

with h5py.File(file, "r") as f:
    for k in [
        "FS_SLV_paramDSD",
        "FS_DSD_paramRDm",
        "FS_PRE_height",
        "FS_PRE_flagPrecip",
        "FS_SLV_precipRateNearSurface",
        "FS_SLV_zFactorFinalNearSurface",
    ]:
        if k in f:
            print(k, f[k].shape, f[k].dtype, dict(f[k].attrs))
        else:
            print(k, "MISSING")
    paramDSD = f["FS_SLV_paramDSD"]
    height = f["FS_PRE_height"]
    import numpy as np

    height_km = height / np.array(1000)
    idx = np.abs(height_km - 2.0).argmin(axis=2)

    s_idx = np.arange(paramDSD.shape[0])[:, None]
    c_idx = np.arange(paramDSD.shape[1])[None, :]

    cand0 = paramDSD[s_idx, c_idx, idx, 0]
    cand1 = paramDSD[s_idx, c_idx, idx, 1]

    print("cand0:", np.nanmin(cand0), np.nanmax(cand0))
    print("cand1:", np.nanmin(cand1), np.nanmax(cand1))

Shape: <KeysViewHDF5 ['nscan', 'nray', 'FS_PRE_flagPrecip', 'FS_PRE_height', 'FS_CSF_typePrecip', 'FS_SLV_precipRateNearSurface', 'FS_SLV_precipRate', 'FS_SLV_zFactorFinal', 'FS_SLV_paramDSD', 'FS_SLV_zFactorFinalNearSurface', 'FS_ScanTime_Minute', 'FS_ScanTime_Year', 'FS_ScanTime_Second', 'FS_ScanTime_Hour', 'FS_ScanTime_Month', 'FS_ScanTime_DayOfMonth', 'FS_FLG_qualityFlag', 'FS_DSD_paramRDm', 'FS_Longitude', 'FS_Latitude', 'nDSD', 'nNode', 'nbin']>
FS_SLV_paramDSD (7931, 49, 176, 2) float32 {'_Netcdf4Coordinates': array([0, 1, 2, 3], dtype=int32), 'DimensionNames': np.bytes_(b'nscan,nray,nbin,nDSD'), 'CodeMissingValue': np.bytes_(b'-9999.9'), 'origname': np.bytes_(b'paramDSD'), 'fullnamepath': np.bytes_(b'/FS/SLV/paramDSD'), 'coordinates': np.bytes_(b'FS_Longitude FS_Latitude nbin nDSD'), '_FillValue': array([-9999.9], dtype=float32), 'DIMENSION_LIST': array([array([<HDF5 object reference>], dtype=object),
       array([<HDF5 object reference>], dtype=object),
       array([<HDF5 ob

TypeError: Only 1D arrays allowed for fancy indexing

In [66]:
def extract_date_from_gpm(filename):
    match = re.search(r"\.(\d{8})-S", filename)
    if match:
        date = match.group(1)
        print(f"\n[DEBUG] Extracted Date: '{date}' from GPM file: '{filename}'")
        return date
    raise ValueError(f"Date not found in {filename}")

def find_merra_file(date):
    print(f"[DEBUG] Searching for MERRA file containing date: '{date}'")
    print(f"[DEBUG] Looking inside folder: {MERRA_FOLDER}")
    
    try:
        files = os.listdir(MERRA_FOLDER)
        print(f"[DEBUG] Total files found in MERRA folder: {len(files)}")
    except FileNotFoundError:
        print(f"[DEBUG] ERROR: The folder {MERRA_FOLDER} does not exist!")
        return None

    for f in files:
        # Ignore hidden system files like .DS_Store
        if f.startswith("."): 
            continue
            
        print(f"   -> Checking against: '{f}'")
        if date in f:
            print(f"      [MATCH] Date '{date}' found inside filename '{f}'!")
            if f.endswith(".nc4"):
                print(f"      [SUCCESS] Extension matches .nc4! Returning file.")
                return os.path.join(MERRA_FOLDER, f)
            else:
                print(f"      [FAIL] Date matched, but file does NOT end with .nc4 (Did it get renamed?)")
                
    print(f"[DEBUG] FAILURE: No matching MERRA file found for date '{date}'.")
    return None

In [46]:
import os
import re
import numpy as np
import xarray as xr
import h5py
import pandas as pd
from scipy.spatial import cKDTree
from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import shutil

GPM_FOLDER = "/Users/utkarshpol/Desktop/project101/data/raw/gpm"
MERRA_FOLDER = "/Users/utkarshpol/Desktop/project101/data/raw/merra"
PROCESSED_FOLDER = "/Users/utkarshpol/Desktop/project101/data/processed"
PARQUET_FOLDER = "/Users/utkarshpol/Desktop/project101/data/parquet"
ZARR_FOLDER = "/Users/utkarshpol/Desktop/project101/data/zarr"

# Ensure output directories exist
os.makedirs(PROCESSED_FOLDER, exist_ok=True)
os.makedirs(PARQUET_FOLDER, exist_ok=True)
os.makedirs(ZARR_FOLDER, exist_ok=True)

def extract_date_from_gpm(filename):
    match = re.search(r"\.(\d{8})-S", filename)
    if match:
        return match.group(1)
    raise ValueError("Date not found")

def find_merra_file(date):
    for f in os.listdir(MERRA_FOLDER):
        if date in f and f.endswith(".nc4"):
            return os.path.join(MERRA_FOLDER, f)
    return None

TREE_CACHE = {}

def build_kdtree(merra_ds):
    lat = merra_ds.lat.values
    lon = merra_ds.lon.values
    lon2d, lat2d = np.meshgrid(lon, lat)
    
    grid_points = np.column_stack((lat2d.ravel(), lon2d.ravel()))
    tree = cKDTree(grid_points)
    return tree, len(lon)

def build_scan_time(f):
    # Adapting to V07 flat keys
    year = f["FS_ScanTime_Year"][:]
    month = f["FS_ScanTime_Month"][:]
    day = f["FS_ScanTime_DayOfMonth"][:]
    hour = f["FS_ScanTime_Hour"][:]
    minute = f["FS_ScanTime_Minute"][:]
    second = f["FS_ScanTime_Second"][:]

    times = np.array([
        np.datetime64(f"{y:04d}-{m:02d}-{d:02d}T{h:02d}:{mi:02d}:{s:02d}")
        for y, m, d, h, mi, s in zip(year, month, day, hour, minute, second)
    ])
    return times

def process_gpm_file(gpm_path):
    global TREE_CACHE

    filename = os.path.basename(gpm_path)
    print(f"\nProcessing: {filename}")

    date = extract_date_from_gpm(filename)
    merra_file = find_merra_file(date)

    if merra_file is None:
        print(f"No MERRA file for date: {date}")
        return

    ###################################
    # LOAD MERRA
    ###################################
    merra = xr.open_dataset(merra_file, chunks={"time": 1})
    aod = merra["TOTEXTTAU"].values
    merra_time = merra.time.values

    ###################################
    # KD TREE CACHE
    ###################################
    if "tree" not in TREE_CACHE:
        tree, lon_len = build_kdtree(merra)
        TREE_CACHE["tree"] = tree
        TREE_CACHE["lon_len"] = lon_len

    tree = TREE_CACHE["tree"]
    lon_len = TREE_CACHE["lon_len"]

    ###################################
    # LOAD DPR
    ###################################
    success = False

    with h5py.File(gpm_path, "r") as f:
        # Basic Variables
        lat = f["FS_Latitude"][:]
        lon = f["FS_Longitude"][:]
        scan_time = build_scan_time(f)

        ##################################
        # TIME EXPAND
        ##################################
        cross = lat.shape[1]
        scan_time2d = np.repeat(scan_time[:, None], cross, axis=1)

        ##################################
        # FLATTEN & SPATIAL MATCH
        ##################################
        lat_flat = lat.ravel()
        lon_flat = lon.ravel()
        scan_flat = scan_time2d.ravel()

        points = np.column_stack((lat_flat, lon_flat))
        _, idx = tree.query(points)

        lat_index = idx // lon_len
        lon_index = idx % lon_len

        ##################################
        # TIME MATCH (Your precise 30 min window)
        ##################################
        merra_time_int = merra_time.astype("datetime64[m]").astype(int)
        scan_time_int = scan_flat.astype("datetime64[m]").astype(int)

        diff = np.abs(scan_time_int[:, None] - merra_time_int)
        time_index = diff.argmin(axis=1)
        time_diff = diff[np.arange(len(time_index)), time_index]

        valid_mask = time_diff <= 30

        ##################################
        # AOD EXTRACTION
        ##################################
        matched_aod = np.full(len(scan_flat), np.nan, dtype=np.float32)
        matched_aod[valid_mask] = aod[time_index[valid_mask], lat_index[valid_mask], lon_index[valid_mask]]
        matched_aod = matched_aod.reshape(lat.shape)

        ################################
        # EXTRACT NEAR SURFACE DPR DATA
        ################################
        R = f["FS_SLV_precipRateNearSurface"][:]
        Z = f["FS_SLV_zFactorFinalNearSurface"][:].astype(float)
        flag = f["FS_PRE_flagPrecip"][:]

        Z[Z < -100] = np.nan

        # Load DSD Profile
        paramDSD_ds = f["FS_SLV_paramDSD"]
        paramDSD = paramDSD_ds[:]
        
        scale = paramDSD_ds.attrs.get("scale_factor", 1.0)
        paramDSD = paramDSD * scale
        paramDSD[paramDSD < -100] = np.nan

        # Create Output Arrays
        Dm = np.full(lat.shape, np.nan, dtype=float)
        DNBw = np.full(lat.shape, np.nan, dtype=float)

        # Dynamic Near Surface Bin Extraction
        # Because binEchoBottom is missing from your keys, we find the lowest valid bin
        nscan, ncross, nbin, _ = paramDSD.shape
        valid_bins_mask = ~np.isnan(paramDSD[:, :, :, 1])

        for i in range(nscan):
            for j in range(ncross):
                valid_bins = np.where(valid_bins_mask[i, j])[0]
                if len(valid_bins) > 0:
                    b = valid_bins[-1] # Grab the lowest valid bin before surface clutter
                    DNBw[i, j] = paramDSD[i, j, b, 0]
                    Dm[i, j] = paramDSD[i, j, b, 1]

        ################################
        # FILTER GOOD RAIN PIXELS & SAVE PARQUET
        ################################
        mask = (flag == 1) & (~np.isnan(Z)) & (R >= 0)
        print(f"Rain Pixels Found: {np.sum(mask)}")

        df = pd.DataFrame({
            "Lat": lat[mask],
            "Lon": lon[mask],
            "R": R[mask],
            "Z": Z[mask],
            "Dm": Dm[mask],
            "DNBw": DNBw[mask],
            "time": scan_time2d[mask].astype(str),
            "AOD": matched_aod[mask]
        })

        pq_save_path = os.path.join(PARQUET_FOLDER, filename.replace(".HDF5.nc4", ".parquet"))
        df.to_parquet(pq_save_path, index=False)
        print(f"Parquet rows: {df.shape[0]} | Saved: {pq_save_path}")

        ################################
        # SAVE 3D/4D DATA TO ZARR
        ################################
        precip_prof = f["FS_SLV_precipRate"][:]
        z_prof = f["FS_SLV_zFactorFinal"][:]
        height_prof = f["FS_PRE_height"][:]

        ds = xr.Dataset(
            data_vars={
                "precipRate": (["scan", "ray", "bin"], precip_prof),
                "zFactor": (["scan", "ray", "bin"], z_prof),
                "height": (["scan", "ray", "bin"], height_prof),
                "Dm_profile": (["scan", "ray", "bin"], paramDSD[:, :, :, 1]),
                "Nw_profile": (["scan", "ray", "bin"], paramDSD[:, :, :, 0]),
                "AOD_matched": (["scan", "ray"], matched_aod)
            },
            coords={
                "lat": (["scan", "ray"], lat),
                "lon": (["scan", "ray"], lon),
                "time": (["scan"], scan_time)
            }
        )
        
        zarr_save_path = os.path.join(ZARR_FOLDER, filename.replace(".HDF5.nc4", ".zarr"))
        ds.to_zarr(zarr_save_path, mode="w")
        print(f"Zarr Saved: {zarr_save_path}")
        
        success = True

    ##################################
    # CLOSE MERRA & MOVE RAW FILE
    ##################################
    merra.close()

    if success:
        new_path = os.path.join(PROCESSED_FOLDER, filename)
        shutil.move(gpm_path, new_path)
        print(f"Moved raw file to: {new_path}")
    else:
        print(f"Processing failed for: {filename}")


def main():
    gpm_files = [
        os.path.join(GPM_FOLDER, f)
        for f in os.listdir(GPM_FOLDER)
        if f.endswith(".nc4")
    ]

    print(f"Total DPR files to process: {len(gpm_files)}")

    for gpm in tqdm(gpm_files):
        try:
            process_gpm_file(gpm)
        except Exception as e:
            print(f"Error processing {gpm}: {e}")

if __name__ == "__main__":
    main()

Total DPR files to process: 1


  0%|          | 0/1 [00:00<?, ?it/s]


Processing: 2A.GPM.Ku.V9-20211125.20150101-S030204-E043435.004784.V07A.HDF5.nc4


/var/folders/bv/gbwgwgl965d0_9tqvkkmnhkc0000gn/T/ipykernel_99310/1379195650.py:77: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 1. This could degrade performance. Instead, consider rechunking after loading.
  merra = xr.open_dataset(merra_file, chunks={"time": 1})


Rain Pixels Found: 19383
Parquet rows: 19383 | Saved: /Users/utkarshpol/Desktop/project101/data/parquet/2A.GPM.Ku.V9-20211125.20150101-S030204-E043435.004784.V07A.parquet


/Users/utkarshpol/Desktop/project101/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Zarr Saved: /Users/utkarshpol/Desktop/project101/data/zarr/2A.GPM.Ku.V9-20211125.20150101-S030204-E043435.004784.V07A.zarr


100%|██████████| 1/1 [00:05<00:00,  5.44s/it]

Moved raw file to: /Users/utkarshpol/Desktop/project101/data/processed/2A.GPM.Ku.V9-20211125.20150101-S030204-E043435.004784.V07A.HDF5.nc4


In [47]:
"""
GPM DPR Processor — Updated for .nc4 flat-variable format (V07)
================================================================
Input  : 2A.GPM.Ku.V9-*.HDF5.nc4  +  MERRA-2 .nc4 AOD files
Output :
  • Parquet  — near-surface (2 km) flat table:
               Lat, Lon, time, R, Z, Dm, dBNw, AOD, typePrecip
  • Zarr     — full 3-D/4-D arrays keyed as (nscan, nray, nbin):
               height, Z_3d, R_3d, Dm_3d, dBNw_3d, aod_matched
               + scalar/2-D metadata: Lat, Lon, time, flagPrecip,
                 typePrecip, aod_matched

Key README facts used
---------------------
paramDSD shape  : (nscan, nray, nbin, 2)
  index 0       : dBNw  = 10·log10(Nw),  Nw in m⁻³ mm⁻¹   (dimensionless dB)
  index 1       : Dm    in mm
missing value   : -9999.9
FS_PRE_height   : (nscan, nray, nbin)  metres above ellipsoid
Near-surface    : bin whose height is closest to 2000 m  (2 km)
flagPrecip      : 1 or 2 = rain detected  (paper uses == 1 for quality)
typePrecip      : leading digit → 1=stratiform, 2=convective
nbin            : 176 (Ku FS), bin index 175 = ellipsoid (ground)
"""

import os
import re
import shutil
import numpy as np
import xarray as xr
import h5py
import pandas as pd
import zarr
from scipy.spatial import cKDTree
from tqdm import tqdm


# ─────────────────────────────────────────────
#  PATHS  — edit to match your directory layout
# ─────────────────────────────────────────────
GPM_FOLDER       = "/Users/utkarshpol/Desktop/project101/data/raw/gpm"
MERRA_FOLDER     = "/Users/utkarshpol/Desktop/project101/data/raw/merra"
PROCESSED_FOLDER = "/Users/utkarshpol/Desktop/project101/data/processed"
PARQUET_FOLDER   = "/Users/utkarshpol/Desktop/project101/data/parquet"
ZARR_FOLDER      = "/Users/utkarshpol/Desktop/project101/data/zarr"

for d in [PROCESSED_FOLDER, PARQUET_FOLDER, ZARR_FOLDER]:
    os.makedirs(d, exist_ok=True)


# ─────────────────────────────────────────────
#  CONSTANTS
# ─────────────────────────────────────────────
FILL_VALUE      = -9999.9        # README §2.2 missing value for paramDSD, R, Z
NEAR_SURF_M     = 2000.0         # 2 km — paper's "near-surface" level
TREE_CACHE      = {}


# ══════════════════════════════════════════════
#  UTILITY HELPERS
# ══════════════════════════════════════════════

def extract_date_from_gpm(filename):
    """Pull YYYYMMDD from filename like 2A.GPM.Ku…YYYYMMDD-S…nc4"""
    match = re.search(r"\.(\d{8})-S", filename)
    if match:
        return match.group(1)
    raise ValueError(f"Date not found in filename: {filename}")


def find_merra_file(date):
    for f in os.listdir(MERRA_FOLDER):
        if date in f and f.endswith(".nc4"):
            return os.path.join(MERRA_FOLDER, f)
    return None


def build_kdtree(merra_ds):
    lat = merra_ds.lat.values
    lon = merra_ds.lon.values
    lon2d, lat2d = np.meshgrid(lon, lat)
    grid_points  = np.column_stack((lat2d.ravel(), lon2d.ravel()))
    tree = cKDTree(grid_points)
    return tree, len(lon)


def build_scan_time(f):
    """
    Build a 1-D datetime64 array of shape (nscan,) from the flat nc4
    ScanTime variables: FS_ScanTime_Year, …_Month, …_DayOfMonth,
    …_Hour, …_Minute, …_Second.
    """
    year   = f["FS_ScanTime_Year"][:]
    month  = f["FS_ScanTime_Month"][:]
    day    = f["FS_ScanTime_DayOfMonth"][:]
    hour   = f["FS_ScanTime_Hour"][:]
    minute = f["FS_ScanTime_Minute"][:]
    second = f["FS_ScanTime_Second"][:]

    times = np.array([
        np.datetime64(
            f"{y:04d}-{mo:02d}-{d:02d}"
            f"T{h:02d}:{mi:02d}:{s:02d}"
        )
        for y, mo, d, h, mi, s in zip(year, month, day, hour, minute, second)
    ])
    return times          # shape (nscan,)


def find_near_surface_bin(height_3d):
    """
    For each (scan, ray) find the bin index whose height is closest to
    NEAR_SURF_M (2000 m).  height_3d shape: (nscan, nray, nbin).
    Returns ns_bin of shape (nscan, nray), dtype int.
    """
    diff = np.abs(height_3d - NEAR_SURF_M)
    ns_bin = np.argmin(diff, axis=2)          # (nscan, nray)
    return ns_bin


# ══════════════════════════════════════════════
#  STAGE 1 — Collocate AOD and write to nc4
#            (matches your original process_gpm_file logic)
# ══════════════════════════════════════════════

def process_gpm_file(gpm_path):
    """
    Opens the flat .nc4 file, collocates MERRA-2 AOD, writes
    'aod_matched' back into the file, then moves it to PROCESSED_FOLDER.
    """
    filename = os.path.basename(gpm_path)
    print(f"\n[AOD] Processing: {filename}")

    date = extract_date_from_gpm(filename)
    merra_file = find_merra_file(date)
    if merra_file is None:
        print(f"  No MERRA file for {date} — skipping")
        return

    # ── Load MERRA ──────────────────────────────────────────────────────
    merra      = xr.open_dataset(merra_file, chunks={"time": 1})
    aod        = merra["TOTEXTTAU"].values          # (time, lat, lon)
    merra_time = merra.time.values

    # ── KD-Tree (cached across files for same MERRA grid) ───────────────
    global TREE_CACHE
    if "tree" not in TREE_CACHE:
        tree, lon_len = build_kdtree(merra)
        TREE_CACHE["tree"]    = tree
        TREE_CACHE["lon_len"] = lon_len
    tree    = TREE_CACHE["tree"]
    lon_len = TREE_CACHE["lon_len"]

    success = False

    with h5py.File(gpm_path, "r+") as f:

        # ── Read lat/lon and scan time ───────────────────────────────────
        lat = f["FS_Latitude"][:]      # (nscan, nray)
        lon = f["FS_Longitude"][:]

        scan_time   = build_scan_time(f)             # (nscan,)
        nscan, nray = lat.shape

        scan_time_2d = np.repeat(
            scan_time[:, None], nray, axis=1
        )                                             # (nscan, nray)

        # ── Flatten for spatial matching ─────────────────────────────────
        lat_flat  = lat.ravel()
        lon_flat  = lon.ravel()
        time_flat = scan_time_2d.ravel()

        _, idx    = tree.query(np.column_stack((lat_flat, lon_flat)))
        lat_index = idx // lon_len
        lon_index = idx %  lon_len

        # ── Time matching (±30 min) ───────────────────────────────────────
        merra_time_int = merra_time.astype("datetime64[m]").astype(int)
        scan_time_int  = time_flat.astype("datetime64[m]").astype(int)

        diff       = np.abs(scan_time_int[:, None] - merra_time_int)
        time_index = diff.argmin(axis=1)
        time_diff  = diff[np.arange(len(time_index)), time_index]
        valid_mask = time_diff <= 30

        # ── AOD extraction ────────────────────────────────────────────────
        matched_aod = np.full(len(time_flat), np.nan, dtype=np.float32)
        matched_aod[valid_mask] = aod[
            time_index[valid_mask],
            lat_index[valid_mask],
            lon_index[valid_mask],
        ]
        matched_aod = matched_aod.reshape(lat.shape)   # (nscan, nray)

        # ── Write AOD back into the file ─────────────────────────────────
        try:
            if "aod_matched" in f:
                del f["aod_matched"]
            f.create_dataset(
                "aod_matched",
                data=matched_aod,
                compression="gzip",
            )
            success = True
            print("  AOD written successfully")
        except Exception as e:
            print(f"  AOD write failed: {e}")

    merra.close()

    if success:
        new_path = os.path.join(PROCESSED_FOLDER, filename)
        shutil.move(gpm_path, new_path)
        print(f"  Moved → {new_path}")
    else:
        print(f"  File NOT moved (AOD failed): {filename}")


# ══════════════════════════════════════════════
#  STAGE 2 — Export Parquet + Zarr
# ══════════════════════════════════════════════

def export_parquet_zarr(gpm_path):
    """
    Reads a processed (AOD-stamped) .nc4 file and exports:

    Parquet — near-surface (2 km) flat table with columns:
              Lat, Lon, time, R, Z, Dm, dBNw, AOD, typePrecip

    Zarr    — full 3-D arrays (nscan, nray, nbin):
                height_m, Z_3d (dBZ), R_3d (mm/h),
                Dm_3d (mm), dBNw_3d (dB)
              2-D arrays (nscan, nray):
                Lat, Lon, flagPrecip, typePrecip, aod_matched
              Coordinate 1-D arrays:
                scan_time (nscan,), nscan_idx, nray_idx
    """
    filename = os.path.basename(gpm_path)
    print(f"\n[Export] {filename}")

    with h5py.File(gpm_path, "r") as f:

        # ────────────────────────────────────────────────────────────────
        # 1. DIMENSIONS & COORDINATES
        # ────────────────────────────────────────────────────────────────
        lat       = f["FS_Latitude"][:]        # (nscan, nray)
        lon       = f["FS_Longitude"][:]       # (nscan, nray)
        scan_time = build_scan_time(f)         # (nscan,)

        nscan, nray = lat.shape

        # ────────────────────────────────────────────────────────────────
        # 2. FLAGS
        # ────────────────────────────────────────────────────────────────
        flag_precip  = f["FS_PRE_flagPrecip"][:]    # (nscan, nray)   int
        type_precip  = f["FS_CSF_typePrecip"][:]    # (nscan, nray)   int32

        # ────────────────────────────────────────────────────────────────
        # 3. HEIGHT (nscan, nray, nbin)
        #    README: 4-byte float, metres above ellipsoid
        # ────────────────────────────────────────────────────────────────
        height = f["FS_PRE_height"][:].astype(np.float32)   # (nscan, nray, nbin)
        nbin   = height.shape[2]

        # ────────────────────────────────────────────────────────────────
        # 4. NEAR-SURFACE BIN INDEX (closest bin to 2 km)
        # ────────────────────────────────────────────────────────────────
        ns_bin = find_near_surface_bin(height)   # (nscan, nray) int

        # ────────────────────────────────────────────────────────────────
        # 5. Z — radar reflectivity profile  (nscan, nray, nbin)
        #    README: 4-byte float, dBZ, fill = -9999.9
        # ────────────────────────────────────────────────────────────────
        Z_3d = f["FS_SLV_zFactorFinal"][:].astype(np.float32)
        Z_3d[Z_3d <= FILL_VALUE + 1] = np.nan       # mask fill values

        # ────────────────────────────────────────────────────────────────
        # 6. R — precipitation rate profile  (nscan, nray, nbin)
        #    README: 4-byte float, mm/hr, fill = -9999.9
        # ────────────────────────────────────────────────────────────────
        R_3d = f["FS_SLV_precipRate"][:].astype(np.float32)
        R_3d[R_3d <= FILL_VALUE + 1] = np.nan

        # ────────────────────────────────────────────────────────────────
        # 7. paramDSD — (nscan, nray, nbin, 2)
        #    index 0 → dBNw = 10·log10(Nw)   [dB]
        #    index 1 → Dm   in mm
        #    fill = -9999.9  (NO additional scale_factor in V07 nc4)
        # ────────────────────────────────────────────────────────────────
        param_dsd = f["FS_SLV_paramDSD"][:].astype(np.float32)

        dBNw_3d = param_dsd[:, :, :, 0].copy()
        Dm_3d   = param_dsd[:, :, :, 1].copy()

        # Mask fill values
        dBNw_3d[dBNw_3d <= FILL_VALUE + 1] = np.nan
        Dm_3d[Dm_3d     <= FILL_VALUE + 1] = np.nan

        # ────────────────────────────────────────────────────────────────
        # 8. AOD — (nscan, nray) written in stage 1
        # ────────────────────────────────────────────────────────────────
        aod_2d = f["aod_matched"][:].astype(np.float32)     # (nscan, nray)

        # ────────────────────────────────────────────────────────────────
        # 9. NEAR-SURFACE EXTRACTION
        #    Fancy-index with ns_bin to pull the 2-km layer
        # ────────────────────────────────────────────────────────────────
        si  = np.arange(nscan)[:, None]          # (nscan, 1)
        ri  = np.arange(nray)[None, :]           # (1, nray)

        Z_ns    = Z_3d[si, ri, ns_bin]           # (nscan, nray)
        R_ns    = R_3d[si, ri, ns_bin]
        Dm_ns   = Dm_3d[si, ri, ns_bin]
        dBNw_ns = dBNw_3d[si, ri, ns_bin]

        # ────────────────────────────────────────────────────────────────
        # 10. RAIN PIXEL MASK  (paper criterion: flagPrecip == 1)
        #     Z and R must also be valid
        # ────────────────────────────────────────────────────────────────
        rain_mask = (
            (flag_precip == 1)
            & (~np.isnan(Z_ns))
            & (~np.isnan(R_ns))
            & (R_ns >= 0)
        )

        print(f"  Rain pixels (near-surface): {np.sum(rain_mask)}")

        # ────────────────────────────────────────────────────────────────
        # 11. BUILD TIME 2-D ARRAY for parquet
        # ────────────────────────────────────────────────────────────────
        scan_time_2d = np.repeat(
            scan_time[:, None], nray, axis=1
        )                                                # (nscan, nray)

    # ─────────────────────────────────────────────────────────────────────
    # PARQUET  — near-surface rain pixels only
    # ─────────────────────────────────────────────────────────────────────
    df = pd.DataFrame({
        "Lat":        lat[rain_mask],
        "Lon":        lon[rain_mask],
        "time":       scan_time_2d[rain_mask].astype(str),
        "R":          R_ns[rain_mask],
        "Z":          Z_ns[rain_mask],
        "Dm":         Dm_ns[rain_mask],
        "dBNw":       dBNw_ns[rain_mask],
        "AOD":        aod_2d[rain_mask],
        "typePrecip": type_precip[rain_mask],
    })

    parquet_path = os.path.join(
        PARQUET_FOLDER,
        filename.replace(".HDF5.nc4", ".parquet").replace(".HDF5", ".parquet"),
    )
    df.to_parquet(parquet_path, index=False, engine="pyarrow")
    print(f"  Parquet rows: {len(df)}  →  {parquet_path}")

    # ─────────────────────────────────────────────────────────────────────
    # ZARR  — full 3-D arrays + 2-D metadata
    # ─────────────────────────────────────────────────────────────────────
    zarr_path = os.path.join(
        ZARR_FOLDER,
        filename.replace(".HDF5.nc4", ".zarr").replace(".HDF5", ".zarr"),
    )

    store = zarr.open(zarr_path, mode="w")

    # Dimensions as attributes for reference
    store.attrs["nscan"]    = int(nscan)
    store.attrs["nray"]     = int(nray)
    store.attrs["nbin"]     = int(nbin)
    store.attrs["source"]   = filename
    store.attrs["near_surface_height_m"] = NEAR_SURF_M
    store.attrs["paramDSD_index0"] = "dBNw = 10*log10(Nw), Nw in m-3 mm-1"
    store.attrs["paramDSD_index1"] = "Dm in mm"

    # ── 1-D coordinate ───────────────────────────────────────────────────
    store.create_dataset(
        "scan_time",
        data=scan_time.astype("datetime64[s]").astype(np.int64),
        chunks=(min(nscan, 512),),
        dtype=np.int64,
        compressor=zarr.Blosc(cname="lz4"),
    )
    store["scan_time"].attrs["units"]       = "seconds since 1970-01-01"
    store["scan_time"].attrs["description"] = "Unix timestamp per scan line"

    # ── 2-D arrays (nscan, nray) ─────────────────────────────────────────
    for name, arr, desc in [
        ("Lat",         lat,         "Latitude  (degrees_north)"),
        ("Lon",         lon,         "Longitude (degrees_east)"),
        ("aod_matched", aod_2d,      "MERRA-2 TOTEXTTAU collocated AOD"),
        ("flagPrecip",  flag_precip.astype(np.int8),
                                     "Rain/NoRain flag: 1=rain(1D), 2=rain(3D)"),
        ("typePrecip",  type_precip.astype(np.int32),
                                     "Precipitation type: leading digit 1=stratiform 2=convective"),
        ("ns_bin_index", ns_bin.astype(np.int16),
                                     "Index of near-surface (2km) bin per pixel"),
    ]:
        store.create_dataset(
            name,
            data=arr,
            chunks=(min(nscan, 256), nray),
            compressor=zarr.Blosc(cname="lz4", clevel=5),
        )
        store[name].attrs["description"] = desc

    # ── 3-D arrays (nscan, nray, nbin) ───────────────────────────────────
    chunk_3d = (min(nscan, 64), nray, nbin)

    for name, arr, units, desc in [
        ("height_m", height,  "m",     "Height above ellipsoid per range bin"),
        ("Z_3d",     Z_3d,    "dBZ",   "Radar reflectivity profile (zFactorFinal)"),
        ("R_3d",     R_3d,    "mm/hr", "Precipitation rate profile"),
        ("Dm_3d",    Dm_3d,   "mm",    "Mass-weighted mean diameter profile (paramDSD index 1)"),
        ("dBNw_3d",  dBNw_3d, "dB",    "10*log10(Nw) profile (paramDSD index 0), Nw in m-3 mm-1"),
    ]:
        store.create_dataset(
            name,
            data=arr,
            chunks=chunk_3d,
            dtype=np.float32,
            compressor=zarr.Blosc(cname="lz4", clevel=5),
        )
        store[name].attrs["units"]       = units
        store[name].attrs["description"] = desc
        store[name].attrs["fill_value"]  = "NaN (original fill -9999.9 replaced)"

    print(f"  Zarr written  →  {zarr_path}")
    print(f"  3-D shape: (nscan={nscan}, nray={nray}, nbin={nbin})")


# ══════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════

def main():

    # ── Stage 1: AOD collocation ──────────────────────────────────────────
    raw_files = sorted([
        os.path.join(GPM_FOLDER, f)
        for f in os.listdir(GPM_FOLDER)
        if f.endswith(".nc4") or f.endswith(".HDF5")
    ])
    print(f"Total raw GPM files: {len(raw_files)}")

    for gpm in tqdm(raw_files, desc="AOD collocation"):
        process_gpm_file(gpm)

    # ── Stage 2: Parquet + Zarr export ───────────────────────────────────
    processed_files = sorted([
        os.path.join(PROCESSED_FOLDER, f)
        for f in os.listdir(PROCESSED_FOLDER)
        if f.endswith(".nc4") or f.endswith(".HDF5")
    ])
    print(f"\nTotal processed files: {len(processed_files)}")

    for gpm in tqdm(processed_files, desc="Parquet+Zarr export"):
        try:
            export_parquet_zarr(gpm)
        except Exception as e:
            print(f"  ERROR on {os.path.basename(gpm)}: {e}")


if __name__ == "__main__":
    main()

Total raw GPM files: 0


AOD collocation: 0it [00:00, ?it/s]



Total processed files: 2


Parquet+Zarr export:   0%|          | 0/2 [00:00<?, ?it/s]


[Export] 2A.GPM.Ku.V9-20211125.20150101-S030204-E043435.004784.V07A.HDF5 2.nc4


Parquet+Zarr export:  50%|█████     | 1/2 [01:05<01:05, 65.86s/it]

  ERROR on 2A.GPM.Ku.V9-20211125.20150101-S030204-E043435.004784.V07A.HDF5 2.nc4: "Unable to synchronously open object (object 'FS_Latitude' doesn't exist)"

[Export] 2A.GPM.Ku.V9-20211125.20150101-S030204-E043435.004784.V07A.HDF5.nc4


Parquet+Zarr export: 100%|██████████| 2/2 [01:09<00:00, 34.65s/it]

  ERROR on 2A.GPM.Ku.V9-20211125.20150101-S030204-E043435.004784.V07A.HDF5.nc4: "Unable to synchronously open object (object 'aod_matched' doesn't exist)"


In [1]:
import os
import re
import numpy as np
import xarray as xr
import h5py
import pandas as pd
import shutil
from scipy.spatial import cKDTree
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

# --- CONFIGURATION ---
GPM_FOLDER = "/Users/utkarshpol/Desktop/project101/data/raw/gpm"
MERRA_FOLDER = "/Users/utkarshpol/Desktop/project101/data/raw/merra"
PROCESSED_FOLDER = "/Users/utkarshpol/Desktop/project101/data/processed"
CSV_FOLDER = "/Users/utkarshpol/Desktop/project101/data/parquet/csv"

# Global cache for the KDTree to avoid rebuilding for every GPM file
TREE_CACHE = {}

def extract_date_from_gpm(filename):
    match = re.search(r"\.(\d{8})-S", filename)
    if match:
        return match.group(1)
    raise ValueError(f"Date not found in filename: {filename}")

def find_merra_file(date):
    for f in os.listdir(MERRA_FOLDER):
        if date in f and f.endswith(".nc4"):
            return os.path.join(MERRA_FOLDER, f)
    return None

def build_kdtree(merra_ds):
    lat = merra_ds.lat.values
    lon = merra_ds.lon.values
    lon2d, lat2d = np.meshgrid(lon, lat)
    grid_points = np.column_stack((lat2d.ravel(), lon2d.ravel()))
    tree = cKDTree(grid_points)
    return tree, len(lon)

def build_scan_time(scan):
    year = scan["Year"][:]
    month = scan["Month"][:]
    day = scan["DayOfMonth"][:]
    hour = scan["Hour"][:]
    minute = scan["Minute"][:]
    second = scan["Second"][:]

    times = np.array([
        np.datetime64(f"{y:04d}-{m:02d}-{d:02d}T{h:02d}:{mi:02d}:{s:02d}")
        for y, m, d, h, mi, s in zip(year, month, day, hour, minute, second)
    ])
    return times

def process_gpm_file(gpm_path):
    filename = os.path.basename(gpm_path)
    print(f"\nProcessing: {filename}")

    try:
        date = extract_date_from_gpm(filename)
    except ValueError as e:
        print(e)
        return

    merra_file = find_merra_file(date)
    if merra_file is None:
        print(f"No MERRA file found for date: {date}")
        return

    # Load MERRA Data
    merra = xr.open_dataset(merra_file, chunks={"time": 1})
    aod = merra["TOTEXTTAU"].values
    merra_time = merra.time.values

    # KDTree Spatial Indexing
    global TREE_CACHE
    if "tree" not in TREE_CACHE:
        tree, lon_len = build_kdtree(merra)
        TREE_CACHE["tree"] = tree
        TREE_CACHE["lon_len"] = lon_len
    
    tree = TREE_CACHE["tree"]
    lon_len = TREE_CACHE["lon_len"]

    success = False

    # Process GPM HDF5
    with h5py.File(gpm_path, "r+") as f:
        fs = f["FS"]
        lat = fs["Latitude"][:]
        lon = fs["Longitude"][:]
        scan = fs["ScanTime"]
        scan_time = build_scan_time(scan)

        # Prepare 2D coordinates and time
        cross = lat.shape[1]
        scan_time2d = np.repeat(scan_time[:, None], cross, axis=1)
        
        lat_flat = lat.ravel()
        lon_flat = lon.ravel()
        scan_flat = scan_time2d.ravel()

        # Spatial Matching
        points = np.column_stack((lat_flat, lon_flat))
        _, idx = tree.query(points)
        lat_index = idx // lon_len
        lon_index = idx % lon_len

        # Temporal Matching (Nearest within 30 mins)
        merra_time_int = merra_time.astype("datetime64[m]").astype(int)
        scan_time_int = scan_flat.astype("datetime64[m]").astype(int)
        
        # Calculate time difference
        diff = np.abs(scan_time_int[:, None] - merra_time_int)
        time_index = diff.argmin(axis=1)
        time_diff = diff[np.arange(len(time_index)), time_index]
        valid_mask = time_diff <= 30

        # Extract and Reshape AOD
        matched_aod = np.full(len(scan_flat), np.nan, dtype=np.float32)
        matched_aod[valid_mask] = aod[
            time_index[valid_mask],
            lat_index[valid_mask],
            lon_index[valid_mask]
        ]
        matched_aod = matched_aod.reshape(lat.shape)

        # Write to HDF5
        try:
            if "aod_matched" in fs:
                del fs["aod_matched"]
            fs.create_dataset("aod_matched", data=matched_aod, compression="gzip")
            success = True
            print("AOD written successfully.")
        except Exception as e:
            print(f"AOD writing failed: {e}")

    merra.close()

    # Move file to processed folder if successful
    if success:
        new_path = os.path.join(PROCESSED_FOLDER, filename)
        shutil.move(gpm_path, new_path)
        print(f"Moved to: {new_path}")
    else:
        print(f"File NOT moved: {filename}")

def export_csv(gpm_path):
    filename = os.path.basename(gpm_path)
    print(f"Exporting CSV: {filename}")

    with h5py.File(gpm_path, "r") as f:
        fs = f["FS"]
        lat = fs["Latitude"][:]
        lon = fs["Longitude"][:]
        R = fs["SLV"]["precipRateNearSurface"][:]
        Z = fs["SLV"]["zFactorFinalNearSurface"][:].astype(float)
        
        # Clean Z values
        Z[Z < -100] = np.nan

        # DSD Processing
        paramDSD_ds = fs["SLV"]["paramDSD"]
        scale = paramDSD_ds.attrs.get("scale_factor", 1.0)
        paramDSD = paramDSD_ds[:] * scale
        
        binBottom = fs["SLV"]["binEchoBottom"][:]
        Dm = np.full(lat.shape, np.nan, dtype=float)
        DNBw = np.full(lat.shape, np.nan, dtype=float)

        nscan, ncross = lat.shape
        for i in range(nscan):
            for j in range(ncross):
                b = binBottom[i, j]
                if 0 < b < paramDSD.shape[2]:
                    DNBw[i, j] = paramDSD[i, j, b, 0]
                    Dm[i, j] = paramDSD[i, j, b, 1]

        # Extra variables
        aod = fs["aod_matched"][:]
        flag = fs["PRE"]["flagPrecip"][:]
        scan_time = build_scan_time(fs["ScanTime"])
        scan_time2d = np.repeat(scan_time[:, None], ncross, axis=1)

        # Mask for rain pixels
        mask = (flag == 1) & (~np.isnan(Z)) & (R >= 0)
        print(f"Rain Pixels found: {np.sum(mask)}")

        # Build DataFrame
        df = pd.DataFrame({
            "Lat": lat[mask],
            "Lon": lon[mask],
            "R": R[mask],
            "Z": Z[mask],
            "Dm": Dm[mask],
            "DNBw": DNBw[mask],
            "time": scan_time2d[mask].astype(str),
            "AOD": aod[mask]
        })

        # Save CSV
        save_path = os.path.join(CSV_FOLDER, filename.replace(".HDF5", ".csv"))
        df.to_csv(save_path, index=False)
        print(f"Saved CSV with {len(df)} rows.")

def main():
    # Ensure folders exist
    for folder in [PROCESSED_FOLDER, CSV_FOLDER]:
        os.makedirs(folder, exist_ok=True)

    # Step 1: Process and Match AOD
    gpm_files = [os.path.join(GPM_FOLDER, f) for f in os.listdir(GPM_FOLDER) if f.endswith(".HDF5")]
    print(f"Total raw GPM files to process: {len(gpm_files)}")
    
    for gpm in tqdm(gpm_files, desc="Processing GPM Files"):
        process_gpm_file(gpm)

    # Step 2: Export to CSV
    processed_files = [os.path.join(PROCESSED_FOLDER, f) for f in os.listdir(PROCESSED_FOLDER) if f.endswith(".HDF5")]
    print(f"\nFiles ready for CSV export: {len(processed_files)}")
    
    for f in processed_files:
        export_csv(f)

if __name__ == "__main__":
    main()

Total raw GPM files to process: 1


Processing GPM Files:   0%|          | 0/1 [00:00<?, ?it/s]


Processing: 2A.GPM.Ku.V9-20211125.20220606-S164656-E181924.046997.V07A.HDF5


Processing GPM Files: 100%|██████████| 1/1 [00:00<00:00,  1.15it/s]

AOD written successfully.
Moved to: /Users/utkarshpol/Desktop/project101/data/processed/2A.GPM.Ku.V9-20211125.20220606-S164656-E181924.046997.V07A.HDF5

Files ready for CSV export: 1
Exporting CSV: 2A.GPM.Ku.V9-20211125.20220606-S164656-E181924.046997.V07A.HDF5


Rain Pixels found: 23746
Saved CSV with 23746 rows.
